# CFD-Drag-2D: CNN surrogate training (M5)

Clones the repository at a fixed commit, runs the surrogate tests, trains the CNN
(mask + Re -> Cd, Cl) on the Kaggle dataset `cfd-drag-2d-lbm`, and evaluates it against the
mean and gradient-boosting baselines on the test split.
Output (`/kaggle/working/m5_run`, plus `m5_run_seed<k>` for extra seeds): `model.pt`, `history.json`, `split.json`, `metrics.json`,
`predictions.csv`, `parity.png`.

In [ ]:
REPO = "https://github.com/PradeepkrishnaGM/CFD-Drag-2D.git"
COMMIT = "main"   # branch or commit SHA
EPOCHS = 150
SEEDS = [0]       # training seeds (weight init, batch order); the split is fixed
RUN = "/kaggle/working/m5_run"
!nvidia-smi --query-gpu=name,memory.total --format=csv
!rm -rf /tmp/repo && git clone -q $REPO /tmp/repo && git -C /tmp/repo checkout -q $COMMIT && git -C /tmp/repo log --oneline -1

In [ ]:
%cd /tmp/repo
!pip install -q -e . --no-deps && pip install -q pytest
import glob, os, torch, sklearn
print("torch", torch.__version__, "cuda", torch.cuda.is_available(), "sklearn", sklearn.__version__)
DATA = os.path.dirname(sorted(glob.glob("/kaggle/input/**/data.csv", recursive=True))[0])
print("data:", DATA)
!python -m pytest -q tests/test_surrogate.py 2>&1 | tail -3

In [ ]:
for seed in SEEDS:
    out = RUN if seed == 0 else f"{RUN}_seed{seed}"
    !python scripts/train.py --data $DATA --out $out --epochs $EPOCHS --device cuda --seed $seed

In [ ]:
for seed in SEEDS:
    out = RUN if seed == 0 else f"{RUN}_seed{seed}"
    !python scripts/evaluate.py --data $DATA --run $out
    !git rev-parse HEAD > $out/commit.txt